# Burned and Unburned Raster Extraction by Time Since Fire

## Import libraries

In [1]:
import arcpy
import os

## Configurations

In [2]:
# GDB and GEE data paths
data_root_dir = r"C:\Users\Huiting Huang\Documents\GIS_Certificate_2025_26\ArcGIS_Projects\EvaluatingForestDisturbances"
gdb_path = os.path.join(data_root_dir, "EvaluatingForestDisturbances.gdb")
gee_landsat_dir = os.path.join(data_root_dir, "Data", "GEE_Landsat")

# Area of interest and processing extent
aoi = "North"  # North, South
extent = f"TNC_AOIs_{aoi}"

# Spatial reference
input_gcs = arcpy.SpatialReference("GCS_WGS_1984")
output_coordinate_system = arcpy.SpatialReference("NAD 1983 UTM Zone 17N")
transformations = arcpy.ListTransformations(
    input_gcs,
    output_coordinate_system,
)
if not transformations:
    raise RuntimeError("No geographic transformation is available.")
geographic_transformation = transformations[0]

## Set geoprocessing environment

In [3]:
# Reset inherited ArcPy environment settings before processing
arcpy.env.workspace = None
arcpy.env.outputCoordinateSystem = None
arcpy.env.geographicTransformations = None
arcpy.env.cellSize = None
arcpy.env.snapRaster = None

# Set ArcPy environment settings for processing
arcpy.env.workspace = gdb_path
arcpy.env.overwriteOutput = True
#arcpy.env.addOutputsToMap = False
arcpy.env.outputCoordinateSystem = output_coordinate_system
arcpy.env.geographicTransformations = geographic_transformation
arcpy.env.extent = extent

## Define analysis inputs and parameters

In [4]:
# Selected years for analysis: 2002(dry), 2003(wet)
# Polygon sets for burn history for the selected years
burn_polygon_set_list = [
    "Burn_Hist_North_1998To2001", # for year 2002
    "Burn_Hist_North_1999To2002", # for year 2003
]

# Vegetation-index rasters for the selected years
# Select parameters: the Landsat sensor, vegetation index, compositing method, and time period.

# Parameter options:
# landsat sensor: L5 (Landsat 5), L7 (Landsat 7), L8 (Landsat 8)
# time period: 06-08 (June–August), 08 (August), 07 (July), 06 (June)
# compositing method: Max, Mean, Median
# vegetation index: 
#   NDVI: normalized difference vegetation index, 
#   NDMI: normalized difference moisture index, 
#   EVI: enhanced vegetation index

dataset = "L5" # Options: L5, L7, L8
composite_strategy = "Max" # Options: Max, Mean, Median
vegetation_index = "EVI" # Options: NDVI, NDMI, EVI
time_period = "06To08" # Options: 06To08, 08, 07, 06

vegetation_index_list = [
    f"{composite_strategy}_{vegetation_index}_2002_{time_period}_North_30m",
    f"{composite_strategy}_{vegetation_index}_2003_{time_period}_North_30m"
]
burn_vegetation_pairs = dict(zip(burn_polygon_set_list, vegetation_index_list))
forest_type_raster = "fortype_aoiN_UTM_30m"

print("Burn polygon and vegetation-index pairs:", burn_vegetation_pairs)

Burn polygon and vegetation-index pairs: {'Burn_Hist_North_1998To2001': 'Max_EVI_2002_06To08_North_30m', 'Burn_Hist_North_1999To2002': 'Max_EVI_2003_06To08_North_30m'}


## Extract rasters for unburned areas for selected years

### Extract forest type rasters for unburned areas

In [ ]:
# Erase burn areas from forest type raster
print("Creating output raster...")
for polygon_set in burn_polygon_set_list:
    year = str(int(polygon_set[-4:])+1)
    output_name = f"{forest_type_raster}_{year}_unburn"

    with arcpy.EnvManager(snapRaster=forest_type_raster, cellSize=forest_type_raster):
        output_raster = arcpy.sa.ExtractByMask(
            in_raster= forest_type_raster,
            in_mask_data=polygon_set,
            extraction_area="OUTSIDE",
            analysis_extent=extent
        )
        output_raster.save(os.path.join(gdb_path, output_name))
    
    print("Created output raster:", output_name)

Creating output raster...
Created output raster: fortype_aoiN_UTM_30m_2002_unburn
Created output raster: fortype_aoiN_UTM_30m_2003_unburn


### Extract vegetation-index rasters for unburned areas

In [ ]:
# Erase burn areas from vegetation index raster
print("Creating output raster...")
for polygon_set, vegetation_index_raster in burn_vegetation_pairs.items():
    output_name = vegetation_index_raster + "_unburn"
    
    with arcpy.EnvManager(snapRaster=forest_type_raster, cellSize=forest_type_raster):
        output_raster = arcpy.sa.ExtractByMask(
            in_raster= vegetation_index_raster,
            in_mask_data=polygon_set,
            extraction_area="OUTSIDE",
            analysis_extent=vegetation_index_raster
        )
        output_raster.save(os.path.join(gdb_path, output_name))

    print("Created output raster:", output_name)

Creating output raster...
Created output raster: Max_EVI_2002_06To08_North_30m_unburn
Created output raster: Max_EVI_2003_06To08_North_30m_unburn


## Extract rasters for burned areas by Time Since Fire (TSF) class for the selected years

### Collect TSF classes from burned areas for each selected year

In [11]:
# Store the unique TSF classes found in each burned-area polygon set.
tsf_values_by_polygon_set = {}
for polygon_set in burn_vegetation_pairs:
    # Read the TimeSinceFire values from the current polygon set.
    tsf_values = []
    with arcpy.da.SearchCursor(polygon_set, ["TimeSinceFire"]) as cursor:
        for row in cursor:
            # Ignore features without a TSF value.
            if row[0] is not None:
                tsf_values.append(row[0])
    # Remove duplicates, sort the classes, and store them by polygon set.
    tsf_values_by_polygon_set[polygon_set] = sorted(set(tsf_values))

### Extract forest type rasters for each TSF class

In [12]:
# Process each burned-area polygon set and its corresponding forest-type raster.
for polygon_set, vegetation_index_raster in burn_vegetation_pairs.items():
    # Derive the analysis year from the polygon-set name.
    year = str(int(polygon_set[-4:]) + 1)
    print("The selected analysis year: ", year)

    # Process each TSF class found in this polygon set.
    print("Creating output raster...")
    
    for tsf in tsf_values_by_polygon_set[polygon_set]:
        # # Create a temporary feature layer containing only the current TSF class.
        # tsf_layer = f"tsf_layer_{tsf}_{year}"
        # where_clause = f"TimeSinceFire = '{tsf}'"
        # arcpy.management.MakeFeatureLayer(polygon_set, tsf_layer, where_clause)

        # Build the output path and remove any existing raster with the same name.
        forest_type_output_name = f"{forest_type_raster}_{year}_{tsf}"
        forest_type_output_path = os.path.join(gdb_path, forest_type_output_name)
        
        # # Extract forest-type raster cells inside the current TSF-class area.
        # with arcpy.EnvManager(snapRaster=forest_type_raster, cellSize=forest_type_raster):
        #     arcpy.sa.ExtractByMask(
        #         in_raster       = forest_type_raster,
        #         in_mask_data    = tsf_layer,
        #         extraction_area = "INSIDE",
        #         analysis_extent = tsf_layer
        #     ).save(forest_type_output_path)
        
        print("Created output raster: ", forest_type_output_name)

        # # Remove the temporary feature layer before processing the next TSF class.
        # arcpy.management.Delete(tsf_layer)

The selected analysis year:  2002
Creating output raster...
Created output raster:  fortype_aoiN_UTM_30m_2002_TSF1
Created output raster:  fortype_aoiN_UTM_30m_2002_TSF2
Created output raster:  fortype_aoiN_UTM_30m_2002_TSF3
Created output raster:  fortype_aoiN_UTM_30m_2002_TSF4
The selected analysis year:  2003
Creating output raster...
Created output raster:  fortype_aoiN_UTM_30m_2003_TSF1
Created output raster:  fortype_aoiN_UTM_30m_2003_TSF2
Created output raster:  fortype_aoiN_UTM_30m_2003_TSF3
Created output raster:  fortype_aoiN_UTM_30m_2003_TSF4


### Extract vegetation-index rasters for each TSF class

In [ ]:
# Process each burned-area polygon set and its corresponding vegetation-index raster.
for polygon_set, vegetation_index_raster in burn_vegetation_pairs.items():
    # Derive the analysis year from the final year in the polygon-set name.
    year = str(int(polygon_set[-4:]) + 1)
    print("The selected analysis year: ", year)

    # Process each TSF class found in this polygon set.
    print("Creating output raster...")

    for tsf in tsf_values_by_polygon_set[polygon_set]:
        # Create a temporary feature layer containing only the current TSF class.
        tsf_layer = f"tsf_layer_{tsf}_{year}"
        where_clause = f"TimeSinceFire = '{tsf}'"
        arcpy.management.MakeFeatureLayer(polygon_set, tsf_layer, where_clause)

        # Build the output path and remove any existing raster with the same name.
        vegetation_index_output_name = f"{vegetation_index_raster}_{tsf}"
        vegetation_index_output_path = os.path.join(gdb_path, vegetation_index_output_name)

        # Extract vegetation-index raster cells inside the current TSF-class area.
        with arcpy.EnvManager(snapRaster=forest_type_raster, cellSize=forest_type_raster):
            arcpy.sa.ExtractByMask(
                in_raster       = vegetation_index_raster,
                in_mask_data    = tsf_layer,
                extraction_area = "INSIDE",
                analysis_extent = tsf_layer
            ).save(vegetation_index_output_path)
        
        print("Created output raster:", vegetation_index_output_name)
            
        # Remove the temporary feature layer before processing the next TSF class.
        arcpy.management.Delete(tsf_layer)

The selected analysis year:  2002
Creating output raster...
Created output raster: Max_EVI_2002_06To08_North_30m_TSF1
Created output raster: Max_EVI_2002_06To08_North_30m_TSF2
Created output raster: Max_EVI_2002_06To08_North_30m_TSF3
Created output raster: Max_EVI_2002_06To08_North_30m_TSF4
The selected analysis year:  2003
Creating output raster...
Created output raster: Max_EVI_2003_06To08_North_30m_TSF1
Created output raster: Max_EVI_2003_06To08_North_30m_TSF2
Created output raster: Max_EVI_2003_06To08_North_30m_TSF3
Created output raster: Max_EVI_2003_06To08_North_30m_TSF4
